In [1]:
import pandas as pd

REVIEWS_CSV = "data/processed/reviews.csv"
CHECKINS_CSV = "data/processed/checkins.csv"
TIPS_CSV = "data/processed/tips.csv"
USERS_CSV = "data/processed/users.csv"
BUSINESSES_CSV = "data/processed/businesses.csv"
OUTPUT_CSV = "data/processed/reviews_per_business.csv"
CHUNKSIZE = 500_000

In [4]:
# --- reviews per business (chunked so we never hold the whole file in memory) ---
def counts_per_entity(entity_id_col:str, entity_csv: str, counted_entity_csv: str, counted_entity:str):

    count_col = f"{counted_entity}_count"
    counts = None

    for chunk in pd.read_csv(counted_entity_csv, usecols=[entity_id_col], chunksize=CHUNKSIZE):
        part = chunk[entity_id_col].value_counts()
        counts = part if counts is None else counts.add(part, fill_value=0)

    df_counted_entity = (
        counts.astype("int64")
        .sort_values(ascending=False)
        .rename_axis(entity_id_col)
        .reset_index(name=count_col)
    )

    # --- join business name/city from businesses.csv ---
    entities = pd.read_csv(
        entity_csv, usecols=[entity_id_col]
    )

    df_counted_entity = df_counted_entity.merge(
        entities, on=entity_id_col, how="left"
    )

    print(f"\nTop 10 entities by {counted_entity} count:")
    print(
        df_counted_entity.head(10)[
            [entity_id_col, count_col]
        ].to_string(index=False)
    )
    total = df_counted_entity[count_col].sum()
    print("Sum: ", total)
    print("Avg: ", total/len(df_counted_entity))

In [10]:
df_business = pd.read_csv(BUSINESSES_CSV)
df_users = pd.read_csv(USERS_CSV)
df_reviews = pd.read_csv(REVIEWS_CSV)
print("Total count of businesses: ", df_business['business_id'].count())
print("Total count of users: ", df_users['user_id'].count())
print("Total count of reviews: ", df_reviews['review_id'].count())

Total count of businesses:  150346
Total count of users:  1000000
Total count of reviews:  1999994


In [5]:
counts_per_entity("business_id", BUSINESSES_CSV, REVIEWS_CSV, "review")


Top 10 entities by review count:
           business_id  review_count
VQcCL9PiNL_wkGf-uF3fjg          4706
GBTPC53ZrG1ZBY3DT8Mbcw          4661
_C7QiQQc47AOEv4PE3Kong          4440
qb28j-FNX1_6xm7u372TZA          3561
DcBLYSvOuWcNReolRVr12A          3217
j-qtdD55OLfSqfsWuQTDJg          2884
ctHjyadbDQAtUFfkcAFEHw          2846
PY9GRfzr4nTZeINf346QOw          2715
U3grYFIeu6RgAAQgdriHww          2689
xlMQBBt9wrtahdqiRDcVSg          2618
Sum:  1999994
Avg:  44.390056597491956


In [6]:
counts_per_entity("business_id", BUSINESSES_CSV, CHECKINS_CSV, "checkin")


Top 10 entities by checkin count:
           business_id  checkin_count
--LC8cIrALInl2vyo701tg              1
zzyx5x0Z7xXWWvWnZFuxlQ              1
---kPU91CF4Lq2-WlRu9Lw              1
--0iUa4sNDFiZFrAdIWhZQ              1
--30_8IhuyMHbSOcNWd6DQ              1
--7PUidqRWpRSpXebiyxTg              1
--7jw19RH9JKXgFohspgQw              1
--8IbOsAAxjKRoYsBFL-PA              1
--9osgUCSDUWUkoTLdvYhQ              1
--ARBQr1WMsTWiwOKOj-FQ              1
Sum:  131930
Avg:  1.0


In [7]:
counts_per_entity("business_id", BUSINESSES_CSV, TIPS_CSV, "tips")


Top 10 entities by tips count:
           business_id  tips_count
FEXhWNCMkv22qG04E83Qjg        1802
-QI8Qi8XWH3D8y8ethnajA         871
Eb1XmmLWyt_way5NNZ7-Pw         696
ytynqOUb3hjKeJfRj5Tshw         659
_ab50qdWOk0DdB6XOrBitw         642
c_4c5rJECZSfNgFj7frwHQ         617
GBTPC53ZrG1ZBY3DT8Mbcw         508
8O35ji_yOMVJmZ6bl96yhQ         485
4i4kmYm9wgSNyF1b6gKphg         444
QHWYlmVbLC3K6eglWoHVvA         441
Sum:  754776
Avg:  7.567662953567884


In [11]:
top = None
for chunk in pd.read_csv(USERS_CSV, usecols=["user_id", "name", "friends"], chunksize=200_000):
    chunk["num_friends"] = chunk["friends"].apply(
        lambda s: 0 if not isinstance(s, str) or s.strip() in ("", "None") else s.count(",") + 1
    )
    part = chunk[["user_id", "name", "num_friends"]]
    top = part if top is None else pd.concat([top, part])
    top = top.nlargest(10, "num_friends")   # keep running top 10 -> flat memory

print(top.to_string(index=False))

               user_id   name  num_friends
qVc8ODYU5SZjKXVBgXdI7w Walker        14995
iLjMdZi0Tm7DQxX1C1_2dg  Ruggy        12395
ZIOCmdFaMIF56FR-nWr_2A  Randy        11026
mV4lknblF-zOKSF8nlGqDA  Scott        10366
Oi1qbcz2m2SnwUeztGYcnQ Steven        10072
hizGc5W1tBHPghM5YKCAtg  Katie         9390
IU86PZPgTDCFwJEuAg2j7g  Danny         9217
n4Y6wdh6om3QtIFlWakfDQ    Rob         8989
djxnI8Ux8ZYQJhiOQkrRhA   Abby         8858
F_5_UNX-wrAFCXuAkBZRDw Rodney         8809


In [41]:
counts_per_entity("user_id", USERS_CSV, REVIEWS_CSV, "review")


Top 10 entities by review count:
               user_id  review_count
_BcWyKQL16ndpBdggh2kNA           894
Xw7ZjaGfr0WNVt6s_5KZfA           522
0Igx-a1wAstiBDerGxXk2A           498
-G7Zkl1wIWBBmD0KRy_sCw           481
bYENop4BuQepBjM1-BI3fA           464
fr1Hz2acAb3OaL3l6DyKNg           407
1HM81n6n4iPIFU5d2Lokhw           406
qjfMBIZpQT9DDtw_BWCopQ           404
wXdbkFZsfDR7utJvbWElyA           402
Um5bfs5DH6eizgjH3xZsvg           399
